# Assessing Accuracy of CFF Global Fits:

## (1): Initializing Requisite Code/Settings:

### (1.1): Import Native Libraries:

In [ ]:
import datetime
import yaml
from pathlib import Path

### (1.2): Import 3rd Party Libraries:

In [ ]:
import numpy as np
from scipy import integrate
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import gepard as g
from gepard.fits import th_KM15

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: gepard version: {g.__version__}")

### (1.4): Customizing Plotting Settings:

In [ ]:
plt.rcParams.update({ "text.usetex": True, "font.family": "serif" })
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 12.0
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 5.0
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['xtick.labelsize'] = 14

plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 12.0
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 5.0
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['savefig.dpi'] = 300
plt.rcParams['axes.labelsize'] = 16

## (2): Data Formatting/Collection Settings:

### (2.1): Versioning:

In [ ]:
with open("closure_test_config.yml", "r") as file:
    config = yaml.safe_load(file)

MAJOR_NUMBER = config["versioning"]["major"]
MINOR_NUMBER = config["versioning"]["minor"]
MAJOR_MINOR_NUMBER = f"{MAJOR_NUMBER}_{MINOR_NUMBER}"

print(f"[INFO]: We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

In [ ]:
K_RANGE = np.linspace(5., 11., 13)
Q2_RANGE = np.linspace(1., 5., 17)
X_B_RANGE = np.linspace(0.1, 0.9, 17)
T_RANGE = np.linspace(-1.0, -0.1, 217)

In [ ]:
nval = 1.35
pval = 1.
nsea = 1.5
rsea = 1.
psea = 2.
bsea = 4.6
Mval = 0.789
rval = 0.918
bval = 0.4
C0 = 2.768
Msub = 1.204
Mtval = 3.993
rtval = 0.881
btval = 0.4
ntval = 0.6
Msea = np.sqrt(0.482)
rpi = 2.646
Mpi = 4.

@np.vectorize
# https://stackoverflow.com/a/77409936 -> for how to integrate over meshgrid domain
def compute_km15_cffs(q_squared, xb, t, k = 0.0):
    xi = xb / (2.0 - xb)
    alpha_val = 0.43 + 0.85 * t
    alpha_sea = 1.13 + 0.15 * t
    Ct = C0 / (1.0 - t / Msub**2)**2

    def fHval(x):
        return (nval * rval / (1 + x) *
                ((2 * x) / (1 + x))**(-alpha_val) *
                ((1 - x) / (1 + x))**bval /
                (1 - ((1 - x) / (1 + x)) * (t / Mval**2))**pval)

    def fHsea(x):
        return (nsea * rsea / (1 + x) *
                ((2 * x) / (1 + x))**(-alpha_sea) *
                ((1 - x) / (1 + x))**bsea /
                (1 - ((1 - x) / (1 + x)) * (t / Msea**2))**psea)

    def fImH(x):
        return np.pi * ((8. / 9.) * fHval(x) + (1. / 9.) * fHsea(x))

    def fPV_ReH(x):
        return -2. * x / (x + xi) * fImH(x)
    
    DR_ReH, _ = integrate.quad(fPV_ReH, 1e-6, 1.0, weight = 'cauchy', wvar = xi)

    real_h_km15 = DR_ReH / np.pi - Ct # Re[H]
    imag_h_km15 = fImH(xi) # Im[H]

    return real_h_km15, imag_h_km15

### (X): Make a meshgrid in $(x_{\textrm{B}}, t, Q^{2})$:

In [ ]:
scrubbed_xb_meshgrid, scrubbed_t_meshgrid = np.meshgrid(X_B_RANGE, T_RANGE, indexing = "ij")

### (X): Evaluate the KM15 CFF Predictions with *one* (or many) kinematic variables fixed:

In [ ]:
_FIXED_Q_SQUARED_VALUE = 1.0

cff_real_h_const_t, cff_imag_h_const_t = compute_km15_cffs(
    q_squared = _FIXED_Q_SQUARED_VALUE,
    xb = scrubbed_xb_meshgrid,
    t = scrubbed_t_meshgrid,
    k = 5.75)

#### (X): **[DEBUGGING]:** A function that generates a random surface so we can test residual plotting methods.

In [ ]:
def random_surface(x, y):
    return 2.* (np.random.rand() * x**2 + np.random.rand() * y**2) * np.exp(-1. * np.random.rand() * x * y )

cff_h_real_global_fit_residual = np.abs(cff_real_h_const_t - random_surface(scrubbed_xb_meshgrid, scrubbed_t_meshgrid))
cff_h_imag_global_fit_residual = np.abs(cff_imag_h_const_t - random_surface(scrubbed_xb_meshgrid, scrubbed_t_meshgrid))

In [ ]:
def inspect_t_lines(dataframe, cff_label = None):

    data = dataframe.copy()

    if cff_label is not None:
        data = data[data["cff"] == cff_label]

    t_line_candidates = (
        data
        .groupby(["k", "xb", "q_squared"])
        .agg(
            number_of_t_points = ("t", "nunique"),
            t_min = ("t", "min"),
            t_max = ("t", "max"),
        )
        .reset_index()
    )

    t_line_candidates = t_line_candidates[
        t_line_candidates["number_of_t_points"] > 1
    ]

    return t_line_candidates.sort_values(
        ["k", "xb", "q_squared"]
    )

In [ ]:
_CFF_PLOT_INFO = {
    "ReH": {
        "label": r"Re$[\mathcal{H}]$",
        "statistics_key": "ReH_pred",
    },
    "ImH": {
        "label": r"Im$[\mathcal{H}]$",
        "statistics_key": "ImH_pred",
    },
    "ReHt": {
        "label": r"Re$[\widetilde{\mathcal{H}}]$",
        "statistics_key": "ReHt_pred",
    },
    "ImHt": {
        "label": r"Im$[\widetilde{\mathcal{H}}]$",
        "statistics_key": "ImHt_pred",
    },
    "ReE": {
        "label": r"Re$[\mathcal{E}]$",
        "statistics_key": "ReE_pred",
    },
    "ImE": {
        "label": r"Im$[\mathcal{E}]$",
        "statistics_key": "ImE_pred",
    },
    "ReEt": {
        "label": r"Re$[\widetilde{\mathcal{E}}]$",
        "statistics_key": "ReEt_pred",
    },
    "ImEt": {
        "label": r"Im$[\widetilde{\mathcal{E}}]$",
        "statistics_key": "ImEt_pred",
    },
}

In [ ]:
cff_global_fitting_results_file = Path(
    f"./hpc/version_{MAJOR_MINOR_NUMBER}"
    f"/cff_summary_statistics_v{MAJOR_MINOR_NUMBER}.csv"
)

cff_global_fitting_results = pd.read_csv(cff_global_fitting_results_file)

In [ ]:
cff_global_fitting_results

In [ ]:
reh_t_lines = inspect_t_lines(
    cff_global_fitting_results,
    cff_label = "ImHt")

print(reh_t_lines)

In [ ]:
reh_t_lines

In [ ]:
for unique_t_value in reh_t_lines:
    print(unique_t_value)

In [ ]:
def plot_cff_vs_t(
    dataframe: pd.DataFrame,
    cff_label: str,
    fixed_k: float,
    fixed_xb: float,
    fixed_q_squared: float,
):

    this_kinematic_set_title_string = (
        rf"$k = {fixed_k:.3f}$ GeV, "
        rf"$x_B = {fixed_xb:.3f}$, "
        rf"$Q^2 = {fixed_q_squared:.3f}$ GeV$^2$"
    )

    data = dataframe[
        (dataframe["cff"] == cff_label)
        & np.isclose(dataframe["k"], fixed_k)
        & np.isclose(dataframe["xb"], fixed_xb)
        & np.isclose(dataframe["q_squared"], fixed_q_squared)
    ].copy()

    data = data.sort_values("t")

    if data.empty:
        raise ValueError(
            f"No data found for {cff_label} with "
            f"k = {fixed_k}, "
            f"xb = {fixed_xb}, "
            f"Q^2 = {fixed_q_squared}."
        )

    if data["t"].nunique() < 2:
        raise ValueError(
            f"Only {data['t'].nunique()} unique t value(s) found. "
            f"At least two are required for a line plot."
        )

    figure, axis = plt.subplots(1, 1, figsize = (10, 8))
    
    axis.errorbar(
        data["t"],
        data["mean"],
        yerr = data["stddev"],
        fmt = "o-",
        capsize = 4.,
        label = "DNN prediction")

    axis.plot(
        data["t"],
        data["km15"],
        linestyle = "--",
        marker = "s",
        label = "KM15")

    axis.set_xlabel(
        r"$t\;[\mathrm{GeV}^2]$",
        fontsize = 15.)

    axis.set_ylabel(
        _CFF_PLOT_INFO[cff_label]["label"],
        fontsize = 15.)

    axis.set_title(
        rf"{_CFF_PLOT_INFO[cff_label]['label']} vs. $t$"
        "\n"
        rf"{this_kinematic_set_title_string}",
        fontsize = 16.)

    axis.legend(fontsize = 15.)

    axis.grid(True, alpha = 0.3)

    figure.tight_layout()

    return figure, axis

In [ ]:
def plot_all_t_trends(dataframe, cff_label):

    # this is a df:
    t_line_candidates = inspect_t_lines(
        dataframe,
        cff_label = cff_label
    )

    print(
        f"[INFO]: Found {len(t_line_candidates)} possible t-trends for {cff_label}."
    )

    for _, candidate in t_line_candidates.iterrows():

        fixed_k = candidate["k"]
        fixed_xb = candidate["xb"]
        fixed_q_squared = candidate["q_squared"]

        print(
            f"[INFO]: Plotting {cff_label} vs. t with "
            f"k = {fixed_k:.6f}, "
            f"x_B = {fixed_xb:.6f}, "
            f"Q² = {fixed_q_squared:.6f} "
            f"using {candidate['number_of_t_points']} points."
        )

        cff_figure, cff_axis = plot_cff_vs_t(
            dataframe,
            cff_label = cff_label,
            fixed_k = fixed_k,
            fixed_xb = fixed_xb,
            fixed_q_squared = fixed_q_squared,
        )

        # [NOTE]: You must ensure this :4f business does not lead to
        # multiple filesnames that are the SAME. 4 figures should be
        # good enough to avoid this degeneracy, but just keep this
        # design decision in mind!
        output_filename = (
            f"cff_{cff_label}_vs_t_k_{fixed_k:.4f}"
            f"_xb_{fixed_xb:.4f}_q2_{fixed_q_squared:.4f}"
            )

        for extension in ["png", "eps"]:
            cff_figure.savefig(
                f"./hpc/version_1_1/{output_filename}.{extension}",
                facecolor = "white",
                transparent = False
            )

        plt.close(cff_figure)

In [ ]:
plot_all_t_trends(
    cff_global_fitting_results,
    cff_label = "ImHt"
)

## Re[$\mathcal{H}$]$(x_{\textrm{B}}, t, Q^{2} = ??)$ Surface Residual Plot:

In [ ]:
#############################################
# figure initialization and customization
#############################################
real_h_residual_plot_figure = plt.figure()
real_h_residual_plot_figure.set_figheight(8)
real_h_residual_plot_figure.set_figwidth(8)

real_h_residual_plot_axis = real_h_residual_plot_figure.add_subplot(projection = '3d')

#############################################
# figure/axis augmentation details:
#############################################
axis_elevation = real_h_residual_plot_axis.elev # extract eleveation param
axis_azimuthal = real_h_residual_plot_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
real_h_residual_plot_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = real_h_residual_plot_axis.transAxes)
real_h_residual_plot_axis.text2D(
    0.01, 0.00, 
    f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = real_h_residual_plot_axis.transAxes)

# Plot the surface.
real_h_residual_plot_axis.plot_surface(
    scrubbed_xb_meshgrid, scrubbed_t_meshgrid, cff_h_real_global_fit_residual,
    cmap = cm.gray, linewidth = 0, antialiased = False)

real_h_residual_plot_axis.set_xlabel(r"$x_{\textrm{B}}$")
real_h_residual_plot_axis.set_ylabel(r"$t$")
real_h_residual_plot_axis.set_title(
    rf"Residuals for Re[$\mathcal{{H}}$] $\left( x_{{\textrm{{B}}}}, t, Q^{{2}} = {_FIXED_Q_SQUARED_VALUE} \ \textrm{{GeV}}^{{2}} \right ) $ (Model vs. True)")

## Im [$\mathcal{H}$]$(x_{\textrm{B}}, t, Q^{2} = ??)$ Surface Residual Plot:

In [ ]:
#############################################
# figure initialization and customization
#############################################
imag_h_residual_plot_figure = plt.figure()
imag_h_residual_plot_figure.set_figheight(8)
imag_h_residual_plot_figure.set_figwidth(8)

imag_h_residual_plot_axis = imag_h_residual_plot_figure.add_subplot(projection = '3d')

#############################################
# figure/axis augmentation details:
#############################################
axis_elevation = imag_h_residual_plot_axis.elev # extract eleveation param
axis_azimuthal = imag_h_residual_plot_axis.azim # extract azimuth parm

# https://matplotlib.org/stable/gallery/mplot3d/text3d.html -> for ax.text2D
imag_h_residual_plot_axis.text2D(
    0.01, 0.03, 
    fr"elevation = {axis_elevation}, $\phi = {axis_azimuthal}^{{\circ}}$", 
    transform = imag_h_residual_plot_axis.transAxes)
imag_h_residual_plot_axis.text2D(
    0.01, 0.00, 
    fr"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
    transform = imag_h_residual_plot_axis.transAxes)

# Plot the surface.
imag_h_residual_plot_axis.plot_surface(
    scrubbed_xb_meshgrid, scrubbed_t_meshgrid, cff_h_imag_global_fit_residual,
    cmap = cm.gray, linewidth = 0, antialiased = False)

imag_h_residual_plot_axis.set_xlabel(r"$x_{\textrm{B}}$")
imag_h_residual_plot_axis.set_ylabel(r"$t$")
imag_h_residual_plot_axis.set_title(
    rf"Residuals for Im[$\mathcal{{H}}$] $\left( x_{{\textrm{{B}}}}, t, Q^{{2}} = {_FIXED_Q_SQUARED_VALUE} \ \textrm{{GeV}}^{{2}} \right ) $ (Model vs. True)")